<a href="https://colab.research.google.com/github/obieshka/Python-2025-/blob/hw_9/%D0%9F%D1%80%D0%B0%D0%BA9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
pip install streamlit pandas scikit-learn numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 99.4 MB/s eta 0:00:00


In [18]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 1. Загрузка данных
df = pd.read_csv(
    'realty_data.csv',
    engine='python',
    on_bad_lines='skip',
    encoding='utf-8',
    sep=','
)

print(f'Размер данных: {df.shape}')
print('Колонки:', df.columns.tolist())
df.head()

Размер данных: (92296, 17)
Колонки: ['product_name', 'period', 'price', 'postcode', 'address_name', 'lat', 'lon', 'object_type', 'total_square', 'rooms', 'floor', 'city', 'settlement', 'district', 'area', 'description', 'source']


,product_name,period,price,postcode,address_name,lat,lon,object_type,total_square,rooms,floor,city,settlement,district,area,description,source
0,"3-комнатная, 137 м²",NaN,63000000,127473.0,"2-й Щемиловский переулок, 5а",55.778894,37.608844,Квартира,137.0,3.0,6.0,Москва,NaN,Тверской район,NaN,Просторная квартира свободной планировки с пан...,ЦИАН
1,"Студия, 16,7 м²",NaN,3250000,108815.0,"Харлампиева, 46",55.551025,37.313054,Квартира,16.7,NaN,1.0,Москва,NaN,Филимонковское поселение,NaN,ВНИМАНИЕ! ОЧЕНЬ ПРИВЛЕКАТЕЛЬНОЕ ПРЕ...,Домклик
2,"3-комнатная, 76 м²",NaN,16004680,NaN,"ЖК Прокшино, 8 к4",55.594802,37.431264,Квартира,76.0,3.0,6.0,Москва,NaN,Сосенское поселение,NaN,"Apт.1684018. 0,01% - гибкая ипотека! Воспользу...",Яндекс.Недвижимость
3,"1-комнатная, 24 м²",NaN,7841776,NaN,"ЖК Прокшино, 6 к2",55.594332,37.428099,Квартира,24.0,1.0,10.0,Москва,NaN,Сосенское поселение,NaN,Продается однокомнатная квартира № 381 в новос...,Новострой-М
4,"3-комнатная, 126 м²",NaN,120000000,121352.0,"Давыдковская, 18",55.721097,37.464342,Квартира,126.0,3.0,16.0,Москва,NaN,Фили-Давыдково район,NaN,Шикарное предложение!\nПродаётся трёхкомнатная...,Домклик


In [19]:
# Целевая переменная
target = 'price'
# Числовые признаки (можно добавить другие, если хотите)
features = ['total_square', 'rooms', 'floor', 'lat', 'lon']

# Проверим наличие всех колонок
missing_cols = [col for col in features + [target] if col not in df.columns]
if missing_cols:
    print('Ошибка: отсутствуют колонки:', missing_cols)
    # Если нет lat/lon, можно обойтись без них
    # Например, если нет lat/lon, убираем их
    features = [f for f in features if f in df.columns]

# Создаём копию данных с нужными колонками
data = df[features + [target]].copy()

# Удаляем строки с пропущенной ценой
data = data.dropna(subset=[target])
print(f'Осталось строк после удаления пропусков в цене: {len(data)}')

# Заполняем пропуски в признаках медианой
for col in features:
    if data[col].isnull().any():
        med = data[col].median()
        data[col].fillna(med, inplace=True)
        print(f'Пропуски в {col} заполнены медианой {med:.2f}')

# Проверим итоговый размер
print('Размер выборки:', data.shape)
data.head()

Осталось строк после удаления пропусков в цене: 92296
Пропуски в rooms заполнены медианой 2.00
Размер выборки: (92296, 6)


/tmp/ipykernel_1533/3318717084.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col].fillna(med, inplace=True)


,total_square,rooms,floor,lat,lon,price
0,137.0,3.0,6.0,55.778894,37.608844,63000000
1,16.7,2.0,1.0,55.551025,37.313054,3250000
2,76.0,3.0,6.0,55.594802,37.431264,16004680
3,24.0,1.0,10.0,55.594332,37.428099,7841776
4,126.0,3.0,16.0,55.721097,37.464342,120000000


In [20]:
X = data[features]
y = data[target]

# Разделение
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Создаём пайплайн: масштабирование + линейная регрессия
model = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

# Обучение
model.fit(X_train, y_train)

# Оценка
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
print(f'R² на тесте: {r2:.4f}')
print(f'MAE на тесте: {mae:,.0f} ₽')

R² на тесте: 0.7314
MAE на тесте: 9,602,494 ₽


In [21]:
# Сохраняем модель и список признаков
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('features.pkl', 'wb') as f:
    pickle.dump(features, f)

print('Модель и признаки сохранены.')

Модель и признаки сохранены.


In [22]:
def load_model():
    with open('model.pkl', 'rb') as f:
        model = pickle.load(f)
    with open('features.pkl', 'rb') as f:
        features = pickle.load(f)
    return model, features

def predict_price(values):
    """
    values: список чисел в порядке features
    """
    model, features = load_model()
    input_data = np.array(values).reshape(1, -1)
    pred = model.predict(input_data)[0]
    return pred

# Проверка на одном примере из теста
sample = X_test.iloc[0].values.tolist()
print('Пример ввода:', sample)
print('Предсказание:', predict_price(sample))
print('Истинная цена:', y_test.iloc[0])

Пример ввода: [39.0, 2.0, 10.0, 55.675369, 37.887693]
Предсказание: 6494916.950982574
Истинная цена: 7200000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [26]:
# Установка streamlit и pyngrok
!pip install streamlit pyngrok -q

# Создание файла app.py обычным способом
with open('app.py', 'w', encoding='utf-8') as f:
    f.write("""
import streamlit as st
import pickle
import numpy as np

st.set_page_config(page_title="Калькулятор стоимости недвижимости")
st.title("🏠 Прогноз стоимости недвижимости")

@st.cache_resource
def load_model():
    with open('model.pkl', 'rb') as f:
        model = pickle.load(f)
    with open('features.pkl', 'rb') as f:
        features = pickle.load(f)
    return model, features

model, features = load_model()

st.sidebar.header("О приложении")
st.sidebar.info(
    "Модель обучена на данных о продажах квартир в Москве. "
    "Введите характеристики объекта и получите прогноз цены."
)

st.subheader("Введите параметры объекта")

input_values = []
cols = st.columns(len(features))
for i, feat in enumerate(features):
    with cols[i]:
        val = st.number_input(
            label=feat.replace('_', ' ').title(),
            value=0.0,
            step=0.1,
            format="%.1f"
        )
        input_values.append(val)

if st.button("Рассчитать стоимость", type="primary"):
    input_array = np.array(input_values).reshape(1, -1)
    pred = model.predict(input_array)[0]
    st.success(f"## 💰 Прогнозируемая цена: {pred:,.0f} ₽")

    with st.expander("Детали ввода"):
        for f, v in zip(features, input_values):
            st.write(f"**{f}**: {v}")

with st.expander("ℹ️ Как это работает"):
    st.markdown(f\"\"\"
    Модель использует **линейную регрессию** на признаках:
    {', '.join(features)}
    Данные масштабируются перед обучением.
    \"\"\")
""")
print("Файл app.py создан")

# Запускаем Streamlit в фоновом процессе
import subprocess
import time
from pyngrok import ngrok

# Запускаем процесс streamlit
process = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Ждём, пока сервер поднимется
time.sleep(5)

# Создаём туннель ngrok
public_url = ngrok.connect(8501)
print(f"✅ Откройте приложение по ссылке: {public_url}")

Файл app.py создан


ERROR:pyngrok.process.ngrok:t=2026-08-21T10:41:07+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-08-21T10:41:07+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
CRITICAL:pyngrok.process.ngrok:t=2026-08-21T10:41:07+0000 lvl=crit msg="command failed" err="authentication failed: This ngrok session is not authenticated. ngrok requi

PyngrokNgrokError: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.